# Final Exam: End-to-End Model Productionisation & Excel Integration

## Q1. Model Deployment (10 Marks)

### Comprehensive Architectural Analysis of Model Deployment Methods

Model deployment represents the critical engineering transition wherein a validated mathematical artifact (machine learning model) is migrated from an isolated experimental sandbox into an active operational environment to deliver real-time or asynchronous business value. Choosing an appropriate deployment method dictates system latency, throughput, infrastructure cost, resilience, and user experience.

---

### 1. Batch Deployment (Offline Inference / Scoring)
- **Mechanics**: The model is scheduled to run periodically (e.g., hourly, nightly, or weekly) over large batches of accumulated data retrieved from data warehouses or object stores. Predictions are persisted into database tables (e.g., PostgreSQL, Snowflake, BigQuery) or distributed caches for downstream consumption.
- **Architectural Workflow**:
  $$\text{Data Warehouse} \longrightarrow \text{Airflow / Prefect DAG} \longrightarrow \text{Model Batch Job} \longrightarrow \text{Prediction Tables / Redis}$$
- **Key Advantages**:
  - Extremely high computational throughput and vectorised parallelism.
  - Zero real-time operational pressure or strict latency constraints (compute can scale opportunistically during off-peak hours).
  - Simple error recovery; jobs can be retried without disrupting end-user interactions.
- **Key Limitations**: Predictions become stale over time; cannot accommodate immediate dynamic customer inputs or intraday behavior shifts.
- **Industry Use Cases**: Overnight retail bank credit risk re-scoring, monthly customer churn probability computation, marketing email propensity scoring.

---

### 2. Streaming & Event-Driven Deployment (Near Real-Time)
- **Mechanics**: Models operate as reactive consumers connected to distributed message brokers (e.g., Apache Kafka, Apache Pulsar, AWS Kinesis). Whenever an event occurs (e.g., a card swipe or ATM withdrawal), the event is published to a topic, the inference service consumes it, runs inference, and emits a prediction event to a downstream topic.
- **Architectural Workflow**:
  $$\text{Event Producer} \xrightarrow{\text{Publish Event}} \text{Kafka Broker} \longrightarrow \text{Streaming Inference Engine} \xrightarrow{\text{Alert / Action}} \text{Kafka Topic}$$
- **Key Advantages**:
  - Fully asynchronous and decoupled architecture; traffic bursts are buffered safely in messaging queues without dropping requests.
  - Seamless integration with complex event processing (CEP) and distributed stream processors (Flink, Spark Streaming).
- **Key Limitations**: Higher architectural complexity; requires distributed state management, topic partition rebalancing, and specialized streaming telemetry.
- **Industry Use Cases**: Algorithmic trading execution, continuous cyber-intrusion detection, real-time IoT anomaly detection.

---

### 3. Serverless Deployment (Function-as-a-Service - FaaS)
- **Mechanics**: Model inference code is deployed as stateless container images or functions onto cloud serverless compute platforms (AWS Lambda, Google Cloud Functions, Azure Functions). The cloud provider automatically provisions compute on demand and scales down to zero when idle.
- **Key Advantages**:
  - Cost efficiency for intermittent, unpredictable workloads (pay strictly per millisecond of compute).
  - Minimal infrastructure maintenance and automated horizontal scaling.
- **Key Limitations**: "Cold start" latency penalties (which can add seconds while container runtimes load weights into memory); strict memory limits ($<10\text{ GB}$) and execution timeout caps.
- **Industry Use Cases**: Internal compliance audit validation tools, back-office document processing pipelines, seasonal tax credit checkers.

---

### 4. Containerised Microservices Orchestration (Docker & Kubernetes)
- **Mechanics**: The model, Python dependencies, and web framework are packaged into an immutable Docker container image and orchestrated across distributed compute clusters using Kubernetes (EKS, GKE, AKS). Horizontal Pod Autoscalers (HPA) scale replicas dynamically based on CPU/memory or request queue depths.
- **Key Advantages**:
  - Environment parity across development, staging, and production.
  - Sophisticated deployment routing strategies:
    - **Canary Deployments**: Route $5\%$ of traffic to a new model version while monitoring error rates before promoting to $100\%$.
    - **Blue-Green Deployments**: Maintain two identical production environments with instant zero-downtime cutover.
    - **Shadow / Dark Deployments**: Replicate production traffic to evaluate a candidate model in parallel without exposing predictions to end users.
- **Key Limitations**: Significant operational overhead and steep engineering learning curve.
- **Industry Use Cases**: High-volume tier-1 core banking infrastructures, global payment networks.

## Q3. FastAPI Deployment with Excel Integration (40 Marks)

### Business Context:
A retail bank wants to predict whether a customer is likely to default on a loan to minimise credit loss while maximising approval volume for creditworthy applicants.

1. Load the `BankLoan` training dataset.
2. Designate `DEFAULTER` as the dependent (target) variable.
3. Utilise all variables except `SN` as independent features.
4. Develop and evaluate a Random Forest classification model with **500 trees**.
5. Load the `BankLoan_Test` dataset and estimate default probabilities.
6. Display the predicted probabilities in an Excel worksheet.
7. Build a high-performance **FastAPI REST API** (`/predict`) to serve the model.
8. Build a macro-enabled Excel workbook (`.xlsm`) with a **working Submit button** that communicates directly with the FastAPI endpoint.
9. Verify and submit all Python source code, model artifacts, and workbook interfaces.

### 3.1 Environment Setup & Library Imports

In [63]:
import numpy as np
import pandas as pd
import joblib

import os
import sys
import json

import matplotlib.pyplot as plt
import seaborn as sns

from main import app
from fastapi.testclient import TestClient
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

### 3.2 Loading and Exploring the BankLoan Training Dataset

Load the `BANK LOAN.csv` dataset containing 700 historical banking customer records and inspect its structure, data types, missing values, and target variable distribution.

In [35]:
data = pd.read_csv("BANK LOAN.csv")

data

,SN,AGE,EMPLOY,ADDRESS,DEBTINC,CREDDEBT,OTHDEBT,DEFAULTER
0,1,3,17,12,9.3,11.36,5.01,1
1,2,1,10,6,17.3,1.36,4.00,0
2,3,2,15,14,5.5,0.86,2.17,0
3,4,3,15,14,2.9,2.66,0.82,0
4,5,1,2,0,17.3,1.79,3.06,1
...,...,...,...,...,...,...,...,...
695,696,2,6,15,4.6,0.26,0.98,1
696,697,1,6,4,11.5,0.37,2.05,0
697,698,2,15,3,7.6,0.49,1.94,0
698,699,3,19,22,8.4,2.30,4.17,0


In [36]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   SN         700 non-null    int64  
 1   AGE        700 non-null    int64  
 2   EMPLOY     700 non-null    int64  
 3   ADDRESS    700 non-null    int64  
 4   DEBTINC    700 non-null    float64
 5   CREDDEBT   700 non-null    float64
 6   OTHDEBT    700 non-null    float64
 7   DEFAULTER  700 non-null    int64  
dtypes: float64(3), int64(5)
memory usage: 43.9 KB


In [38]:
print("\nTarget Variable (DEFAULTER) Distribution:")
print(data['DEFAULTER'].value_counts(normalize= False))
print(data['DEFAULTER'].value_counts(normalize= True).round(4) * 100)


Target Variable (DEFAULTER) Distribution:
DEFAULTER
0    517
1    183
Name: count, dtype: int64
DEFAULTER
0    73.86
1    26.14
Name: proportion, dtype: float64


### Descriptive Statistics of Bank Loan Features
Inspecting statistical measures (mean, standard deviation, percentiles, min, and max) across all independent variables:

In [39]:
data.describe()

,SN,AGE,EMPLOY,ADDRESS,DEBTINC,CREDDEBT,OTHDEBT,DEFAULTER
count,700.00000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000
mean,350.50000,1.902857,8.388571,8.278571,10.260571,1.553457,3.058229,0.261429
std,202.21688,0.765301,6.658039,6.824877,6.827234,2.117209,3.287524,0.439727
min,1.00000,1.000000,0.000000,0.000000,0.400000,0.010000,0.050000,0.000000
25%,175.75000,1.000000,3.000000,3.000000,5.000000,0.370000,1.047500,0.000000
50%,350.50000,2.000000,7.000000,7.000000,8.600000,0.855000,1.985000,0.000000
75%,525.25000,2.000000,12.000000,12.000000,14.125000,1.905000,3.927500,1.000000
max,700.00000,3.000000,31.000000,34.000000,41.300000,20.560000,27.030000,1.000000


| Variable Name |  Data Type   |    Role    |                   Business Description & Unit                    |
|:-------------:|:------------:|:----------:|:----------------------------------------------------------------:|
|    **SN**     |   Integer    | Identifier |         Serial customer number (**Excluded from model**)         |
|    **AGE**    |   Integer    |  Feature   |         Age bracket category (1= <30, 2= 30-45, 3= >45)          |
|  **EMPLOY**   |   Integer    |  Feature   |          Years with current employer (tenure stability)          |
|  **ADDRESS**  |   Integer    |  Feature   |   Years at current residential address (geographic stability)    |
|  **DEBTINC**  |    Float     |  Feature   | Debt-to-income ratio (Total monthly debt / Monthly income in %)  |
| **CREDDEBT**  |    Float     |  Feature   |      Total credit card debt balance in thousands of dollars      |
|  **OTHDEBT**  |    Float     |  Feature   |   Total other consumer and auto loans in thousands of dollars    |
| **DEFAULTER** | Binary (0/1) |   Target   |  1= Customer defaulted on loan, 0= Customer repaid successfully  |

## 3.3 Feature Definition & Train-Validation Splitting

As specified in the assignment tasks:
- **Target Variable**: `DEFAULTER`
- **Independent Variables**: All variables except `SN` (`AGE`, `EMPLOY`, `ADDRESS`, `DEBTINC`, `CREDDEBT`, `OTHDEBT`).

Perform an 80/20 stratified train-validation split to assess out-of-sample generalisation performance.

In [40]:
feature_cols = [c for c in data.columns if c not in ['SN', 'DEFAULTER']]

In [41]:
X = data[feature_cols]
y = data['DEFAULTER']

In [42]:
# 80/20 Stratified Split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size= 0.20,
    random_state= 42,
    stratify= y
)

print(f"Training Subset Size:   {X_train.shape[0]} Records")
print(f"Validation Subset Size: {X_val.shape[0]} Records")

Training Subset Size:   560 Records
Validation Subset Size: 140 Records


## 3.4 Random Forest Classifier Training (500 Trees) & Validation Benchmark

Train a **RandomForestClassifier** with `n_estimators= 500` as required by the assignment guidelines and evaluate performance across key classification metrics.

In [46]:
rf_val = RandomForestClassifier(
    n_estimators= 500,
    random_state= 42
)
rf_val.fit(X_train, y_train)

# Predictions and probabilities
y_val_pred = rf_val.predict(X_val)
y_val_prob = rf_val.predict_proba(X_val)[:, 1]

In [44]:
# Metrics calculation
acc = accuracy_score(y_val, y_val_pred)
roc_auc = roc_auc_score(y_val, y_val_prob)
prec = precision_score(y_val, y_val_pred)
rec = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
cm = confusion_matrix(y_val, y_val_pred)

In [27]:
print("RANDOM FOREST VALIDATION PERFORMANCE (500 TREES)")
print(f"Accuracy:                    {acc:.4f} ({acc*100:.2f}%)")
print(f"Area Under ROC Curve (AUC):  {roc_auc:.4f}")
print(f"Precision (Defaulter):       {prec:.4f}")
print(f"Recall (Defaulter):          {rec:.4f}")
print(f"F1-Score (Defaulter):        {f1:.4f}")

RANDOM FOREST VALIDATION PERFORMANCE (500 TREES)
Accuracy:                    0.8286 (82.86%)
Area Under ROC Curve (AUC):  0.8305
Precision (Defaulter):       0.6970
Recall (Defaulter):          0.6216
F1-Score (Defaulter):        0.6571


In [50]:
print("Confusion Matrix: \n", cm)


print(f"\n[TN={cm[0,0]:3d}  FP={cm[0,1]:3d}]")
print(f"[FN={cm[1,0]:3d}  TP={cm[1,1]:3d}]")

Confusion Matrix: 
 [[93 10]
 [14 23]]

[TN= 93  FP= 10]
[FN= 14  TP= 23]


In [52]:
print("Detailed Classification Report:\n")
print(classification_report(y_val, y_val_pred, target_names= ["Non-Defaulter (0)", "Defaulter (1)"]))

Detailed Classification Report:

                   precision    recall  f1-score   support

Non-Defaulter (0)       0.87      0.90      0.89       103
    Defaulter (1)       0.70      0.62      0.66        37

         accuracy                           0.83       140
        macro avg       0.78      0.76      0.77       140
     weighted avg       0.82      0.83      0.83       140



## 3.5 Full Model Training on Entire Dataset (500 Trees) & Feature Importance

Now train the final production model utilising all 700 historical customer records with 500 trees and analyse the relative importance of each credit feature.

In [54]:
rf_full = RandomForestClassifier(
    n_estimators= 500,
    random_state= 42
)

rf_full.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [56]:
# Feature Importances
importances = pd.Series(rf_full.feature_importances_, index= feature_cols).sort_values(ascending= False)

fi_df = pd.DataFrame({
    'Feature': importances.index,
    'Importance_Score': importances.values.round(4),
    'Importance_Pct': [f"{v*100:.2f}%" for v in importances.values]
})

print(fi_df.to_string(index=False))

# Serialize model
joblib.dump(rf_full, "model.joblib")
print(f"\nModel successfully serialised to: model.joblib ({os.path.getsize('model.joblib'):,} bytes)")

 Feature  Importance_Score Importance_Pct
 DEBTINC            0.2495         24.95%
CREDDEBT            0.2098         20.98%
  EMPLOY            0.1910         19.10%
 OTHDEBT            0.1784         17.84%
 ADDRESS            0.1331         13.31%
     AGE            0.0383          3.83%

Model successfully serialised to: model.joblib (9,337,449 bytes)


## 3.6 Evaluating BankLoan_Test Dataset & Estimating Default Probability

Load the `BANK LOAN_TEST.csv` dataset (10 customer records), extract features, estimate the probability of default via `predict_proba(X_test)[:, 1]`, and evaluate against actual borrower outcomes.

In [57]:
test_data = pd.read_csv("BANK LOAN_TEST.csv")

In [58]:
X_test = test_data[feature_cols]
test_probs = rf_full.predict_proba(X_test)[:, 1]
test_preds = rf_full.predict(X_test)

In [59]:
# Assemble results table
test_results = test_data.copy()
test_results['ESTIMATED_PROB_DEFAULT'] = np.round(test_probs, 4)
test_results['PROBABILITY_PCT'] = [f"{p*100:.2f}%" for p in test_probs]
test_results['PREDICTED_CLASS'] = test_preds
test_results['RISK_TIER'] = [
    'High Risk' if p >= 0.50 else ('Moderate Risk' if p >= 0.30 else 'Low Risk')
    for p in test_probs
]
test_results['CORRECT'] = test_results['PREDICTED_CLASS'] == test_results['DEFAULTER']

In [60]:
print("="*95)
print("     BANKLOAN_TEST ESTIMATED PROBABILITY OF DEFAULT & PREDICTIONS")
print("="*95)

display_cols = ['SN', 'AGE', 'EMPLOY', 'ADDRESS', 'DEBTINC', 'CREDDEBT', 'OTHDEBT',
                'DEFAULTER', 'PROBABILITY_PCT', 'PREDICTED_CLASS', 'RISK_TIER', 'CORRECT']
print(test_results[display_cols].to_string(index= False))

     BANKLOAN_TEST ESTIMATED PROBABILITY OF DEFAULT & PREDICTIONS
 SN  AGE  EMPLOY  ADDRESS  DEBTINC  CREDDEBT  OTHDEBT  DEFAULTER PROBABILITY_PCT  PREDICTED_CLASS     RISK_TIER  CORRECT
  1    3      16       10      9.1     11.30     5.01          1          67.80%                1     High Risk     True
  2    1       8        5     17.0      1.46     4.00          0          31.60%                0 Moderate Risk     True
  3    2      15       14      5.5      0.86     2.17          0           0.00%                0      Low Risk     True
  4    3      13       13      2.9      2.56     0.82          0           8.60%                0      Low Risk     True
  5    1       2        1     16.5      1.79     3.06          1          78.40%                1     High Risk     True
  6    3       5        6     10.2      0.39     2.16          0           6.60%                0      Low Risk     True
  7    2      18        8     31.1      3.80    16.67          0          28.60%       

In [62]:
test_acc = accuracy_score(test_results['DEFAULTER'], test_preds)

print(f"Test Set Evaluation Accuracy: {test_acc*100:.2f}% ({test_results['CORRECT'].sum()} of {len(test_results)} correct classifications)")

Test Set Evaluation Accuracy: 100.00% (10 of 10 correct classifications)


## 3.7 FastAPI REST API Implementation (`main.py`) & Automated Testing

The model is deployed as a high-performance REST API using **FastAPI** (`main.py`). The service exposes:
- `POST /predict`: Real-time single customer loan assessment (optimized for Excel VBA integration).
- `POST /predict-batch`: Batch inference for multiple customer profiles.
- `GET /test-predictions`: Serves the 10 test dataset predictions with evaluation metrics.
- `GET /health`: Health and telemetry check.
- `GET /interactive-tester`: Browser GUI for instant manual testing.

We test the live FastAPI application using `fastapi.testclient.TestClient`:

In [64]:
client = TestClient(app)

# 1. Health check verification
res_health = client.get("/health")
print("1. Health Endpoint (/health):", res_health.status_code, res_health.json())

1. Health Endpoint (/health): 200 {'status': 'healthy', 'model_loaded': True, 'model_type': 'RandomForestClassifier', 'n_estimators': 500, 'features_expected': ['AGE', 'EMPLOY', 'ADDRESS', 'DEBTINC', 'CREDDEBT', 'OTHDEBT']}


In [68]:
# 2. Test Customer 1 from BankLoan_Test (Actual Defaulter)
cust1_payload = {
    "AGE": 3, "EMPLOY": 16, "ADDRESS": 10,
    "DEBTINC": 9.1, "CREDDEBT": 11.3, "OTHDEBT": 5.01
}
res_pred1 = client.post("/predict", json= cust1_payload)
print("\n 2. Customer 1 Prediction Response:")
print(json.dumps(res_pred1.json(), indent= 2))


 2. Customer 1 Prediction Response:
{
  "default_probability": 0.678,
  "default_probability_pct": "67.80%",
  "prediction": 1,
  "prediction_label": "Defaulter",
  "risk_tier": "High Risk",
  "recommendation": "Decline / Reject Application",
  "input_features": {
    "AGE": 3.0,
    "EMPLOY": 16.0,
    "ADDRESS": 10.0,
    "DEBTINC": 9.1,
    "CREDDEBT": 11.3,
    "OTHDEBT": 5.01
  }
}


In [69]:
# 3. Test Customer 2 from BankLoan_Test (Actual Non-Defaulter)
cust2_payload = {
    "AGE": 1, "EMPLOY": 8, "ADDRESS": 5,
    "DEBTINC": 17.0, "CREDDEBT": 1.46, "OTHDEBT": 4.00
}
res_pred2 = client.post("/predict", json= cust2_payload)
print("\n 3. Customer 2 Prediction Response:")
print(json.dumps(res_pred2.json(), indent= 2))


 3. Customer 2 Prediction Response:
{
  "default_probability": 0.316,
  "default_probability_pct": "31.60%",
  "prediction": 0,
  "prediction_label": "Non-Defaulter",
  "risk_tier": "Moderate Risk",
  "recommendation": "Manual Underwriter Review Required",
  "input_features": {
    "AGE": 1.0,
    "EMPLOY": 8.0,
    "ADDRESS": 5.0,
    "DEBTINC": 17.0,
    "CREDDEBT": 1.46,
    "OTHDEBT": 4.0
  }
}


## 3.8 Excel Integration & Macro-Enabled Workbook (`Bank_Loan_Default_Prediction.xlsm`)

### Architecture of the Excel-to-FastAPI System
```
+-----------------------------------------------------------------------------------------+
|                  EXCEL MACRO-ENABLED WORKBOOK (.xlsm) FRONTEND                          |
+-----------------------------------------------------------------------------------------+
|  [Single Customer Prediction Sheet]                                                     |
|    - C6: Age Category (1, 2, 3)                                                         |
|    - C7: Employment Tenure (Years)                                                      |
|    - C8: Address Tenure (Years)                                                         |
|    - C9: Debt-to-Income Ratio (%)                                                       |
|    - C10: Credit Card Debt ($k)                                                         |
|    - C11: Other Debt ($k)                                                               |
|                                                                                         |
|    +-----------------------------------------------+                                    |
|    |  [BUTTON]: "Submit to FastAPI (/predict)"     |                                    |
|    +-----------------------+-----------------------+                                    |
|                            | Click Event triggers VBA Sub SubmitLoanPrediction()        |
+----------------------------|------------------------------------------------------------+
                             |
                             v
+-----------------------------------------------------------------------------------------+
|                           VBA MACRO COMMUNICATION LAYER                                 |
+-----------------------------------------------------------------------------------------+
|  #If Mac Then                                                                           |
|      MacScript("do shell script 'curl -s -X POST -H ...'")                              |
|  #Else                                                                                  |
|      Set xmlHttp = CreateObject("MSXML2.ServerXMLHTTP.6.0")                             |
|      xmlHttp.Open "POST", "http://127.0.0.1:8000/predict", False                        |
|      xmlHttp.send jsonPayload                                                           |
|  #End If                                                                                |
+----------------------------|------------------------------------------------------------+
                             |  HTTP POST /predict  (JSON Payload)
                             v
+-----------------------------------------------------------------------------------------+
|                              FASTAPI BACKEND SERVICE                                    |
+-----------------------------------------------------------------------------------------+
|  1. Ingests JSON payload -> Validates against Pydantic CustomerLoanFeatures             |
|  2. Transforms into DataFrame -> Calls model.predict_proba(df)[:, 1]                    |
|  3. Evaluates credit risk tier & decision recommendations                               |
|  4. Returns JSON: {"default_probability": 0.3160, "risk_tier": "Moderate Risk", ...}    |
+----------------------------+------------------------------------------------------------+
                             |
                             |  HTTP 200 OK Response (JSON)
                             v
+-----------------------------------------------------------------------------------------+
|                    EXCEL WORKSHEET CELL & UI UPDATES                                    |
+-----------------------------------------------------------------------------------------+
|  - C14: Predicted Default Probability -> Formatted as Percentage (e.g., 31.60%)         |
|  - C15: Risk Assessment Tier -> "Moderate Risk (Underwriter Review)"                    |
|  - C16: Credit Decision Recommendation -> "Manual Underwriter Review Required"          |
|  - C17: Backend API Status -> "HTTP 200 OK (FastAPI)" (Green)                           |
|  - C18: Timestamp -> Current System Timestamp                                           |
|  - Popup Alert: Instant MessageBox displaying full credit evaluation summary            |
+-----------------------------------------------------------------------------------------+
```

### Verification of Generated Artifacts
We verify that all required project deliverables have been generated and are present in the assignment directory:

In [70]:
artifacts = [
    ("BANK LOAN.csv", "Historical Training Dataset (700 rows)"),
    ("BANK LOAN_TEST.csv", "Test Evaluation Dataset (10 rows)"),
    ("model.joblib", "Serialized 500-Tree Random Forest Model Artifact"),
    ("main.py", "FastAPI Application Source Code"),
    ("vba_macro.bas", "Exported VBA Macro Module Source Code"),
    ("Bank_Loan_Default_Prediction.xlsm", "Macro-Enabled Excel Workbook with Submit Button"),
    ("requirements.txt", "Project Dependencies & Versions"),
    ("render.yaml", "Cloud Deployment Configuration")
]

In [72]:
print("     PROJECT DELIVERABLES & SUBMISSION VERIFICATION \n")

for filename, desc in artifacts:
    exists = os.path.exists(filename)
    size = f"{os.path.getsize(filename):,} bytes" if exists else "MISSING"
    status_str = "PASS [FOUND]" if exists else "FAIL [MISSING]"
    print(f"[{status_str}]  {filename:<35} | {size: <15} | {desc}")

     PROJECT DELIVERABLES & SUBMISSION VERIFICATION 

[PASS [FOUND]]  BANK LOAN.csv                       | 19,402 bytes    | Historical Training Dataset (700 rows)
[PASS [FOUND]]  BANK LOAN_TEST.csv                  | 318 bytes       | Test Evaluation Dataset (10 rows)
[PASS [FOUND]]  model.joblib                        | 9,337,449 bytes | Serialized 500-Tree Random Forest Model Artifact
[PASS [FOUND]]  main.py                             | 20,537 bytes    | FastAPI Application Source Code
[PASS [FOUND]]  vba_macro.bas                       | 12,687 bytes    | Exported VBA Macro Module Source Code
[PASS [FOUND]]  Bank_Loan_Default_Prediction.xlsm   | 31,999 bytes    | Macro-Enabled Excel Workbook with Submit Button
[PASS [FOUND]]  requirements.txt                    | 234 bytes       | Project Dependencies & Versions
[PASS [FOUND]]  render.yaml                         | 243 bytes       | Cloud Deployment Configuration
